# Week 3

> **📌 이번 실습은 가급적이면 Colab에서 진행해주세요!**
>
> Mission2에서 사용하는 sLLM(`Qwen2.5-3B-Instruct`)은 CPU 환경에서 실행하면 답변을 생성하는 데 오래 걸리기 때문이에요. Colab의 GPU를 사용하면 몇 초만에 답변을 확인할 수 있어요.
>
> 오른쪽 상단 ▼ 버튼을 눌러 **런타임 유형 변경 → 하드웨어 가속기 T4 GPU** 로 바꿔주세요.
(아래 사진처럼 표시되면 변경 완료입니다!)

<img src="imgs/colab_t4_gpu.png" width="600" />

Week1~2에서 우리는 직접 모델을 설계하고, 학습까지 해보았습니다.

그런데 이런 의문이 들 수 있습니다.

> 굳이 처음부터 모델을 만들어야 할까..? 
> 이미 엄청난 양의 데이터를 학습해서, **언어를 잘 이해하는 똑똑한 모델**이 있는데, 그 모델을 **그대로 가져와서** 우리가 풀고 싶은 문제에 맞게 **조금만 다시 가르치면 되지 않을까?**

2주차에서는 직접 모델을 설계했다면, 남은 3, 4주차에서는 이미 학습된 모델을 가져와 우리 데이터로 추가로 학습시키는 파인튜닝(Fine-Tuning)을 해볼 것입니다. 
이번 주에는 4주차에서 fine-tuning에 사용할 모델인 **BERT의 구조**를 분석해볼 거에요. 

이에 더해, 다음과 같은 생각이 들 수 있습니다.

> “모델이 학습할 때 배우지 않은 최신 정보를 답하게 하고 싶은데..”

이 문제를 해결하기 위해 등장한 방식이 RAG입니다.


이번 주차에서는 다음 두 가지를 학습해보아요!

1. BERT 구조
2. RAG


[4주 흐름](../README.md)과 [실행 환경 안내](../docs/runtime.md)를 먼저 확인해주세요.

### 이번 주차의 학습 목표


- 이미 학습된 모델을 가져다 쓰는 **사전학습 모델(Pretrained Model)** 의 개념을 알아봅니다.
- **BERT 모델의 구조**를 살펴봅니다.
- **RAG(Retrieval-Augmented Generation)** 의 전체 흐름을 이해합니다.
- **Fine-Tuning과 RAG의 차이**를 비교해봅니다.


### 파일 구조

```text
gdg-7th-ai-mission/
├── week1/
│   ├── README.md
│   └── week1_mission.ipynb
├── week2/
│   ├── README.md
│   └── week2_mission.ipynb
├── week3/
│   ├── README.md
│   └── week3_mission.ipynb
├── docs/runtime.md
├── artifacts/
│   └── week1/
│       └── week1_state.json
├── pyproject.toml
├── uv.lock
└── README.md
```


### ⚓️ Mission

이번 실습에서는 **준비 코드는 제공하고, 사전학습 모델을 불러오는 핵심 동작을 직접 작성**합니다.

- **[제공 코드]** 는 그대로 실행해주세요.
- **[⚓️ 미션]** 에서는 TODO 안내에 따라 `None`을 코드로 바꿔주세요.
- 각 단계의 **실행 결과 확인**에서 출력을 직접 확인해주세요.
- **⭐️ 개념 정리**와 **생각해보기**는 아래의 답변 셀에 작성해주세요.
- 코드를 수정했다면 해당 셀부터 아래로 다시 실행해주세요.

이 노트북에는 정답과 실행 출력이 없습니다. 미션을 채운 뒤 위에서부터 실행해주세요.

이번 주차부터는 Keras 대신 Hugging Face의 **`transformers`** 라이브러리와 **PyTorch**를 사용합니다. 

두 패키지가 환경에 없다면 설치가 필요합니다. 로컬에서는 미션 폴더 루트의 터미널에서 아래 명령을 실행해주세요.

```bash
uv add transformers torch
```

웹 Colab에서는 아래 셀에서 설치 줄의 주석을 해제해주세요

In [ ]:
# 웹 Colab 이용자 분들은 아래 주석을 해제하고 패키지를 설치해주세요. 
# %pip install transformers torch

#### 1. 사전학습 모델 준비하기

우리가 상품 리뷰를 긍정/부정으로 분류하는 AI를 만들고 싶다고 가정해봅시다.

그런데 직접 모델을 처음부터 학습시키려면:

- 엄청 많은 데이터
- 긴 학습 시간
- 큰 GPU 자원

이 필요합니다.

하지만 이미 한국어를 이해하는 BERT 모델이 존재합니다.

그러면 우리는:

> “한국어는 이미 알고 있으니까, 상품 리뷰만 조금 더 학습시키자!”

라는 전략을 사용할 수 있습니다.

이것이 Fine-Tuning입니다.


**⭐️ 개념 정리**

- 사전학습 모델(Pretrained Model)이란 무엇인지 정리해주세요.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


아래 코드에서는 사용할 한국어 BERT 모델을 가져와요. KLUE/bert-base는 한국 NLP 연구팀이 뉴스, 국민청원, 나무위키, 모두의 말뭉치 등 대규모 한국어 데이터로 사전학습한 BERT 모델입니다.  

모델 파일(약 440MB)은 처음 불러올 때 한 번만 내려받습니다.



In [ ]:
# [제공 코드] 라이브러리를 불러오고 사용할 모델을 지정합니다.
import sys

import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = 'klue/bert-base'  # 한국어 BERT 모델
SEED = 42

torch.manual_seed(SEED)
print('Python version:', sys.version)
print('Transformers version:', transformers.__version__)
print('PyTorch version:', torch.__version__)


#### 2. Tokenizer 이해하기

week1에서 모델은 문장을 직접 읽지 못하니 숫자로 바꿔주어야 하고, 이 역할을 하는 친구가 Tokenizer였습니다.

Week1에서는 훈련 데이터로 단어 사전을 직접 만들었습니다(`adapt`). 이번에는 모델이 사전학습할 때 사용한 사전을 함께 불러옵니다. 모델이 알고 있는 번호 체계를 그대로 사용해야 하기 때문이죠! 

아래 문장을 tokenizer에 넣고 결과를 확인해보아요.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) # klue/bert-base 모델과 짝이 되는 tokenizer를 불러옵니다.

sample_text = '이 상품 정말 마음에 든다'

encoded = tokenizer(sample_text)

print(encoded)


**⭐️ 개념 정리**

- 출력 결과에서 `input_ids`는 무엇일까요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


**[⚓️ 미션]** `input_ids`의 숫자가 실제로 어떤 토큰인지 확인해 `tokens`의 빈칸을 채워 숫자를 토큰으로 되돌려주세요.

**HINT:** `tokenizer.convert_ids_to_tokens()`를 이용해 input_ids를 토큰 문자열 리스트로 변환할 수 있어요. 


In [ ]:
tokens = None  # TODO: input_ids를 토큰 문자열 리스트로 바꿔주세요.

print(encoded['input_ids'])
print(tokens)



**실행 결과 확인**

토큰이 `[CLS]`로 시작해 `[SEP]`로 끝나는지 확인해주세요. 이는 우리가 넣지 않았지만 tokenizer가 추가한 특수 토큰으로, `[CLS]`는 문장의 시작을, `[SEP]`는 문장의 끝을 표시합니다.

`##`이 붙은 6번째 토큰이 보이네요. `##`은 앞 토큰에 이어지는 조각이라는 뜻이에요.


#### 3. Fine-Tuning 모델 불러오기

이번에는 감정 분석을 위한 모델을 불러와봅시다.

기존 BERT는 “한국어를 이해하는 AI”이지만, 우리는 “한국어를 이해한 뒤, **긍정/부정을 분류**하는 AI”가 필요합니다. 그래서 마지막에 `긍정/부정 분류기`를 붙여주어야 합니다.

`AutoModelForSequenceClassification`은 사전학습된 BERT에 분류기를 붙여주는 클래스입니다.

**[⚓️ 미션]** 사전학습 모델을 불러오는 코드를 작성해주세요.

- 위에서 tokenizer를 불러온 코드와 같이 `from_pretrained()`를 사용합니다.

- 긍정/부정 두 가지로 분류하도록 `num_labels`를 지정해주세요.

**HINT:** `AutoModelForSequenceClassification.from_pretrained(모델 이름, num_labels=분류할 종류의 수)` 형태로 작성할 수 있어요. ([Hugging Face: 모델](https://huggingface.co/learn/llm-course/ko/chapter2/3))


In [ ]:
model = None  # TODO: MODEL_NAME의 사전학습 모델을 불러와주세요.

print('전체 파라미터 수:', f'{model.num_parameters():,}')


**실행 결과 확인**

**생각해보기:**

- 출력된 파라미터 수를 week2 `model.summary()`의 `Total params`와 비교해보아요. 이를 근거로, 왜 모델을 처음부터 만들지 않고 이미 학습된 모델을 가져다 쓰는지 설명해주세요.
- 왜 `num_labels=2`일까요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


#### 4. 모델 구조 확인하기

**Transformer란?**

우리가 사용할 BERT는 Transformer 모델입니다. Transformer 모델이 무엇인지, 그 역사를 간단하게 설명드리겠습니다. 개념이 어려우니 지금 전부 이해하지 못해도 괜찮습니다.

초기의 Seq2Seq 모델은 Encoder-Decoder 구조였고, 내부에 RNN이나 LSTM을 사용했습니다. Encoder는 입력 문장의 정보를 하나의 Context Vector로 압축하고, Decoder는 이 Context Vector를 입력으로 받아 출력을 생성합니다. 그러나 문장이 길수록 앞선 과거 정보를 현재 예측에 활용하지 못하는 장기 의존성 문제(Long-term dependency problem)가 존재했습니다.

2014년 Bahdanau 외 연구진이 제안한 Attention 메커니즘은 Decoder가 출력을 만들 때 Encoder의 모든 시점 정보를 참고하도록 했습니다. 다만 한 문장 내에서 각 단어들의 관계를 보는 것이 아니라, 인코더-디코더 간의 관계(cross-attention)를 보는 구조였습니다.

2017년, 구글은 "Attention Is All You Need" 논문을 발표합니다. 이 논문에서 제안된 Transformer는 기존 Seq2Seq의 문제를 Self-Attention으로 해결합니다. Self-Attention은 한 문장 내에서 각 단어들의 관계를 계산하는 방식으로, 이번 주차에 사용할 BERT도 Transformer의 Encoder를 기반으로 만들어진 모델입니다.

Transformer를 더 공부하고 싶다면 아래 영상을 참고해주세요.

📺 [3Blue1Brown, 트랜스포머, ChatGPT가 트랜스포머로 만들어졌죠. - DL5](https://youtu.be/g38aoGttLhI?si=OJwT07kdcH7hfaZ0)

    Week2 마지막에 추천드린 영상입니다. 트랜스포머 안에서 문장이 처리되는 전체 흐름을 시각적으로 보여줍니다.

Week2에서는 `model.summary()`로 모델 구조를 확인했습니다. 이번에는 `print(model)`로 BERT의 구조를 출력해봅시다.


In [ ]:
# [제공 코드] 모델 구조를 출력합니다.
print(model)


모델 구조를 출력해보면 매우 복잡한 출력 결과를 확인해볼 수 있을 거예요.

먼저 BERT가 문장을 어떻게 처리하는지 이해해봅시다.

BERT는 문장을 이렇게 처리합니다.

```text
“이 상품 마음에 든다”
↓
문장을 숫자로 바꿈
↓
단어들의 관계를 살펴봄
↓
문장 전체 의미를 정리함
↓
긍정인지 부정인지 판단함
```

이 흐름을 실제 모델 구조와 연결해보면,

```text
1. 문장을 숫자로 바꿈
→ embeddings: BertEmbeddings

2. 단어들의 관계를 살펴봄
→ encoder: BertEncoder
→ attention: BertAttention
→ self: BertSelfAttention

3. 문장 전체 의미를 정리함
→ pooler: BertPooler

4. 긍정인지 부정인지 판단함
→ classifier: Linear(in_features=768, out_features=2, bias=True)
```




##### 4-1. Embedding: 문장을 숫자로 바꾸는 단계

모델 구조 출력 결과에서 아래 부분을 찾아보세요.

```text
(embeddings): BertEmbeddings
```

Transformer 모델은 문장을 글자 그대로 처리하지 못합니다.

그래서 먼저 단어를 숫자 벡터로 변환합니다.

예를 들어 `"오늘 날씨 좋다"`라는 문장은 모델 안에서 `[0.12, -0.44, 0.91, ...]` 와 같은 숫자 형태로 바뀝니다. 이 과정을 Embedding이라고 합니다. Week2의 `Embedding` 레이어가 이 역할을 했었죠.

출력 코드에서 아래 부분들을 찾아보세요.

```text
word_embeddings -> 단어 뜻을 숫자로 바꿈
position_embeddings -> 단어 위치를 숫자로 바꿈
token_type_embeddings -> 문장 구분 정보를 숫자로 바꿈
```

화살표 옆의 간단한 설명을 참고하여, 각각의 embedding layer가 어떠한 역할을 하는지 정리해주세요.

**⭐️ 개념 정리**

- `word_embeddings`, `position_embeddings`, `token_type_embeddings` 의 역할에 대해서 정리해주세요.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


##### 4-2. Attention: 단어 간 관계를 이해하는 단계

모델 구조 출력 결과에서 아래 부분을 찾아보세요.

```text
(attention): BertAttention
(self): BertSelfAttention
```

Attention은 문장 속 단어들이 서로 얼마나 관련 있는지 계산하는 구조입니다.

예를 들어 `"오늘 날씨가 좋아서 기분이 좋다"`라는 문장에서 `날씨 ↔ 좋다`, `기분 ↔ 좋다` 같은 관계를 함께 봅니다. 즉, 단어를 따로따로 이해하지 않고, 단어 간 관계, 문맥, 문장 전체 흐름을 함께 고려합니다.

이렇게 문맥을 이해하는 것이 매우 중요한 또 다른 이유가 있습니다. 예를 들어, `배를 먹는다`, `배를 탄다` 두 문장 모두 `배`라는 단어가 등장하지만 의미는 다릅니다. 첫 번째 문장에서 `배`는 과일을 의미하고, 두 번째 문장에서 `배`는 탈것을 의미합니다. BERT는 `배`라는 단어만 따로 보는 것이 아니라, 주변에 있는 `먹는다`, `탄다` 같은 단어들과의 관계를 함께 보며 현재 문맥에서 어떤 의미로 사용되었는지 이해합니다.

이처럼 Transformer 모델은 단어 자체만 보는 것이 아니라, **단어 간 관계와 문맥을 함께 고려하여 문장의 의미를 이해**합니다.

**⭐️ 개념 정리**

- Attention 구조에 대해서 정리해주세요.

**참고 영상**

아래 영상을 참고하시면 이해에 도움이 될 것 같아 올려보아요! 

- 📺 [GDG 홍익대학교, Introduction To Attention Algorithm](https://www.youtube.com/watch?v=7LVRciBEGaM): Attention 개념을 설명하는 세미나 영상입니다.
- 📺 [3Blue1Brown, 그 이름도 유명한 어텐션, 이 영상만 보면 이해 완료! - DL6](https://youtu.be/_Z3rXeJahMs?si=-rYqkkIkCCfjnOy4): 단어 벡터가 문맥에 따라 바뀌는 과정을 시각적으로 보여주는 영상입니다.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


##### 4-3. Query / Key / Value 이해하기

Attention 안에는 아래 구조가 있습니다.

```text
query:
현재 단어가 찾고 싶은 정보

key:
각 단어가 가진 특징

value:
실제로 참고할 정보
```

Query, Key, Value는 Attention이 단어 간 관계를 계산할 때 사용하는 요소입니다. 현재 단어가 문장 안의 다른 단어들 중 어떤 단어를 얼마나 참고할지 계산하는 과정이라고 이해할 수 있습니다.

즉, Query / Key / Value는 단어들이 서로를 참고하면서 문맥을 이해할 수 있도록 도와주는 구조입니다.

**⭐️ 개념 정리**

- Query, Key, Value는 어떤 역할을 하나요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


##### 4-4. Layer: 문장을 점점 깊게 이해하는 단계

모델 구조 출력 결과에서 아래 부분을 찾아보세요.

```text
(0-11): 12 x BertLayer
```

이 뜻은 BERT 안에 Transformer Layer가 총 12개 있다는 의미입니다.

Transformer는 문장을 한 번만 읽고 끝내지 않습니다. 여러 layer를 거치며 문장을 점점 더 깊게 이해하는데요. 즉, Transformer는 layer를 반복적으로 거치며 문장의 의미를 점점 정교하게 분석합니다.


##### 4-5. Hidden Size 이해하기

모델 구조에서 768이라는 숫자가 반복해서 등장합니다.

이 숫자는 하나의 토큰을 몇 개의 숫자로 표현하는지를 의미합니다. Week2의 `EMBEDDING_DIM = 128`과 비슷한 설정입니다.

예를 들어 BERT에서 hidden size가 `768`이면, `배`라는 토큰 하나를 모델 안에서는 숫자 768개짜리 벡터 형태로 표현한다는 뜻입니다. 앞에서 출력한 `벡터 크기`도 같은 의미입니다.

`배 → [0.12, -0.44, 0.91, ...]  총 768개`

그렇다면 왜 단어를 숫자 하나가 아니라 여러 개의 숫자로 표현할까요?

그 이유는 단어 하나에도 다양한 정보가 담겨야 하기 때문입니다.

예를 들어 단어에는 의미, 문맥, 감정, 문장 안에서의 역할, 다른 단어와의 관계들 같은 정보들이 포함될 수 있습니다.

즉, 단어 하나에도 의미, 문맥, 감정, 문장 안에서의 역할 등 다양한 정보가 담길 수 있기 때문에, Transformer 모델은 단어를 숫자 하나가 아니라 hidden size 크기의 벡터로 표현합니다.


##### 4-6. Pooler: 문장 전체 의미 만들기

모델 구조 출력 결과에서 아래 부분을 찾아보세요.

```text
(pooler): BertPooler
```

지금까지는 저희가 단어 단위 정보 위주로 처리하는 단계를 설명드렸습니다. 하지만, 많은 작업은 문장 전체 의미가 필요합니다.

예를 들어, `디자인은 좋았지만 배송은 너무 느렸다` 이러한 리뷰가 있다고 가정해봅시다. 이 문장은 단어 하나만 보면 긍정인지 부정인지 판단하기 어렵습니다.

따라서 Transformer는 여러 단어 정보를 종합하여 문장 전체 의미를 하나의 벡터로 정리하는 과정이 필요합니다.

이 역할을 하는 부분이 `Pooler`입니다. `BertPooler`는 12개의 layer를 거치며 문맥이 반영된 `[CLS]` 토큰의 벡터를 입력으로 받아, `dense`와 `activation`을 거친 뒤 문장 벡터로 내보냅니다.

**⭐️ 개념 정리**

- Pooler의 역할을 정리해주세요.

**생각해보기:**

- Week2에서 사용한 Pooling과 `BertPooler`는 문장 벡터를 만드는 방식이 어떻게 다른가요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


##### 4-7. Classifier: 최종 작업 수행하기

모델 구조 출력 결과에서 아래 부분을 찾아보세요.

```text
(classifier): Linear(in_features=768, out_features=2, bias=True)
```

지금까지는 Transformer 기반 모델이 문장을 어떻게 이해하는지를 설명드렸습니다. 이번 실습의 최종 목표는 긍정/부정을 분류하는 것입니다.

마지막 classifier가 이러한 분류 작업을 수행합니다. 앞서 `num_labels`로 지정한 판단기가 바로 이 부분입니다.

- Transformer 본체: 문장을 이해하는 역할
- classifier: 문제를 해결하는 역할

이라고 볼 수 있습니다.


**⭐️ 개념 정리**

지금까지 살펴본 내용을 바탕으로 정리해주세요.

- BERT 모델 구조를 간단하게 정리해주세요.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


#### 5. BERT로 리뷰 분류해보기

지금까지 살펴본 BERT에 실제 리뷰를 넣어 분류해봅시다. '디자인은 좋았지만 배송이 너무 느렸어요' 리뷰를 모델에 입력으로 넣은 뒤, 각 단계에서 나온 결과의 `shape`를 확인해봐요.



In [ ]:
# [제공 코드] 리뷰 하나가 각 layer를 지나면서, 어떤 모양으로 바뀌는지 출력합니다.
review = '디자인은 좋았지만 배송이 너무 느렸어요'
inputs = tokenizer(review, return_tensors='pt')

with torch.no_grad():
    bert_output = model.bert(**inputs)
    logits = model(**inputs).logits

print('토큰:', tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))
print('input_ids:', tuple(inputs['input_ids'].shape))
print('last_hidden_state:', tuple(bert_output.last_hidden_state.shape))
print('pooler_output:', tuple(bert_output.pooler_output.shape))
print('logits:', tuple(logits.shape))


**실행 결과 확인**

맨 앞의 `1`은 문장 개수(배치 크기)입니다. 토큰 수를 `N`이라고 하면, 전체 구조는 다음과 같습니다. 출력된 `shape`와 비교해보세요.

```text
정수 토큰 N개
   (1, N)
     ↓ embeddings + encoder (12 x BertLayer)
   (1, N, 768)   → 토큰마다 768차원 벡터
     ↓ pooler
   (1, 768)      → 문장을 대표하는 벡터 하나
     ↓ classifier
   (1, 2)        → 부정, 긍정 각각의 점수
```

입력을 벡터로 바꾸고, 문장 벡터 하나로 압축한 뒤, 분류 결과를 만든다는 점에서 Week2 모델의 흐름과 비슷합니다.

이번에는 이 모델로 리뷰의 감정을 예측해보겠습니다. 아래 코드는 `logits`를 softmax로 0~1 사이의 점수로 바꿔 출력합니다. Week1과 같이 부정은 `0`, 긍정은 `1`번 자리입니다.


In [ ]:
# [제공 코드] 아직 파인튜닝하지 않은 모델로 리뷰의 감정을 예측해봅시다.
example_sentences = [
    '배송도 빠르고 품질도 정말 좋아요',
    '완전 별로예요. 돈이 아까워요',
    '디자인은 좋았지만 배송이 너무 느렸어요',
]

for sentence in example_sentences:
    inputs = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        scores = torch.softmax(model(**inputs).logits, dim=-1)[0]
    print(f'문장: {sentence}')
    print(f'부정 점수: {scores[0]:.4f}, 긍정 점수: {scores[1]:.4f}')


**생각해보기:**

- 출력된 점수가 리뷰의 감정과 잘 맞나요? 이 점수를 믿어도 될지, 앞서 3단계에서 본 `classifier` 안내를 근거로 설명해주세요.
- 이 모델을 상품 리뷰 분류에 사용하려면 앞으로 무엇을 더 해야 할까요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


# Mission2 (RAG)

Fine-Tuning은 모델 자체를 다시 학습시키는 방식이였습니다.

하지만 Fine-Tuning만으로는 해결하기 어려운 문제도 존재합니다.

사전학습 모델은 학습 당시의 데이터만 알고 있기 때문에, 최신 정보나 외부 문서를 자동으로 알 수는 없습니다.

이 문제를 해결하기 위해 등장한 방식이 바로 `RAG (Retrieval-Augmented Generation)`입니다. 

RAG란 질문과 관련된 외부 문서를 먼저 검색한 뒤, 그 문서를 함께 참고하여 답변하는 방식입니다. 



**참고 자료**

제가 RAG 기술을 이해할 때 도움을 많이 받았던 테디노트님의 강의입니다.

- 📺 [테디노트, EP01. RAG의 동작 과정 쉽게 이해하기!](https://www.youtube.com/watch?v=zybyszetEcE) >> **해당 영상에서 4분 44초까지 RAG(검색 증강 생성)의 개념을 정말 쉽게 설명해주시니, 꼭 시청하는 걸 추천드려요!** 이후에는 문서를 벡터 DB에 저장하기 전에 거쳐야 하는 전처리 과정(문서 로드 → 분할 → 임베딩)을 다룹니다. 관심 있는 분들은 들어보세요. 

추후 프로젝트나 개인 공부로 RAG를 만들어보고 싶다면, 테디노트님의 LangChain 무료 전자책과 GitHub도 확인해보세요.(LangChain은 RAG 같은 LLM 애플리케이션을 만들 때 사용하는 프레임워크입니다. 다만 이번 실습에서 LangChain은 사용하지 않습니다.)

- [<랭체인LangChain 노트> - LangChain 한국어 튜토리얼](https://wikidocs.net/book/14314)
- [langchain-kr GitHub](https://github.com/teddylee777/langchain-kr)

## RAG 문장 처리 방식

```markdown
질문 입력
↓
질문을 벡터로 변환
↓
질문과 비슷한 문서 검색
↓
검색된 문서를 Context로 생성
↓
질문 + Context를 함께 모델에 입력
↓
최종 결과 생성
```

### **1. 문장 임베딩 모델과 벡터 검색 라이브러리 준비하기**


In [ ]:
import sys
!uv pip install --python {sys.executable} sentence-transformers faiss-cpu

# 웹 Colab 이용자 분들은 위 두 줄을 주석 처리하고, 아래 주석을 해제해서 패키지를 설치해주세요!
# %pip install sentence-transformers faiss-cpu

이번 실습에서는 두가지 라이브러리를 설치하도록 하겠습니다. 

- sentence-transformers: 문장을 벡터로 변환
- faiss: 벡터를 빠르게 검색

하기 위해 사용하는 라이브러리입니다.

### **2. RAG에 사용할 라이브러리 불러오기**


In [ ]:
# [제공 코드] RAG에 사용할 라이브러리를 불러옵니다.
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


### **3. Knowledge Base 만들기**

이번 실습에서는 쇼핑몰의 **배송/환불 정책 문서**를 참고해 고객 질문에 답하는 **상품 Q&A 챗봇**을 만들어보겠습니다.

먼저 정책 문서를 `policy.txt` 파일로 저장합니다.


In [ ]:
%%writefile policy.txt
[배송 기간] 결제 완료 후 1~3 영업일 이내에 출고되며, 출고 후 보통 1~2일 안에 도착합니다. 주말과 공휴일은 출고되지 않습니다.

[배송비] 30,000원 이상 구매 시 무료배송이며, 30,000원 미만 주문은 배송비 3,000원이 부과됩니다.

[도서산간 배송] 제주도 및 도서산간 지역은 무료배송 주문이더라도 추가 배송비 3,000원이 부과됩니다.

[주문 취소] 상품이 출고되기 전에는 마이페이지에서 바로 주문을 취소할 수 있습니다. 출고된 이후에는 취소가 불가능하며 반품 절차로 진행됩니다.

[교환/반품 신청] 상품 수령 후 7일 이내에 마이페이지에서 교환 또는 반품을 신청할 수 있습니다. 단순 변심으로 인한 반품은 왕복 배송비 6,000원을 고객이 부담합니다.

[반품 불가] 상품을 사용한 흔적이 있거나, 택(tag)을 제거했거나, 상품 포장이 훼손된 경우에는 교환 및 반품이 불가능합니다.

[불량/오배송] 불량 상품이나 주문과 다른 상품을 받은 경우 수령 후 30일 이내에 신청할 수 있으며, 배송비는 판매자가 부담합니다.

[환불 처리] 반품 상품이 물류센터에 도착해 검수가 완료되면 3 영업일 이내에 환불됩니다. 카드 결제의 경우 카드사 사정에 따라 환불 반영까지 3~5 영업일이 추가로 소요될 수 있습니다.


In [ ]:
# [제공 코드] 정책 문서를 chunk 단위로 나눕니다.
with open("policy.txt", encoding="utf-8") as f:
    text = f.read()

knowledge_base = [p.strip() for p in text.split("\n\n") if p.strip()]

print(len(knowledge_base))
print(knowledge_base[0])


RAG에서는 모델이 참고할 외부 문서가 필요합니다. 이러한 외부 문서 저장소를 
`Knowledge Base` 이라고 부릅니다. 즉, 모델이 학습한 정보만을 사용하는 것이 아니라, 필요한 정보를 외부에서 추가로 찾아 참고하는 구조입니다. 

긴 문서를 통째로 검색하면 질문과 관련 없는 내용까지 함께 딸려오기 때문에, 문서를 작은 단위로 나눠서 저장합니다. 이렇게 문서를 나눈 조각을 `chunk`라고 부릅니다. 저희 코드에서는 빈 줄(`\n\n`)을 기준으로 문단 단위로 나누어, 총 8개의 chunk를 Knowledge Base로 사용합니다.

**⭐️ 개념 정리**

- Knowledge Base는 왜 필요한가요?

**생각해보기:**

- 문서를 chunk 단위로 나누는 이유는 무엇일까요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


### **4. 문장을 벡터로 변환하기**


In [ ]:
# [제공 코드] 문서를 벡터로 변환합니다.
embedder = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2'
)

kb_embeddings = embedder.encode(
    knowledge_base,
    convert_to_numpy=True
)


RAG는 문장을 그대로 비교하지 않습니다.

대신 문장을 숫자 벡터로 변환한 뒤, 벡터 간 거리를 비교하여 의미가 비슷한 문서를 찾습니다.

예를 들어 `"돈은 언제 돌려받을 수 있나요?"`와 `"반품 상품의 검수가 완료되면 3 영업일 이내에 환불됩니다."`는 겹치는 단어가 거의 없지만 의미는 비슷합니다.

문장 임베딩 모델은 이런 의미 정보를 벡터 공간에 반영합니다.

**⭐️ 개념 정리**

- 의미가 비슷한 문장은 벡터 공간에서 어떻게 나타날까요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


### **5. 벡터 검색 인덱스 만들기**


In [ ]:
# [제공 코드] 벡터 검색 인덱스를 만듭니다.
dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(kb_embeddings)


RAG에서는 질문과 가장 비슷한 문서를 빠르게 찾아야 합니다.

문서가 몇 개 없다면 하나씩 비교해도 괜찮지만, 문서가 수천 개, 수만 개, 수백만 개로 많아지면 모든 문서를 직접 비교하는 방식은 너무 오래 걸립니다.

그래서 FAISS를 사용하여 문서 벡터들을 검색하기 좋은 형태로 저장합니다.

이때 만들어지는 검색용 구조를 `index`라고 부릅니다. 쉽게 말하면, `index`는 문서 벡터들을 빠르게 찾을 수 있도록 정리해둔 검색용 보관함입니다.

저희 코드에서는 L2 distance를 기준으로 비슷한 문서를 찾는 FAISS 검색기를 만드는 코드입니다.

**⭐️ 개념 정리**

- L2 distance는 무엇을 의미하나요?



**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


### **6. 질문과 비슷한 문서 검색하기**


In [ ]:
# [제공 코드] 질문과 가장 가까운 문서 top_k개를 검색합니다.
def retrieve(query, top_k=2):
    query_vec = embedder.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_vec, top_k)

    return [knowledge_base[i] for i in indices[0]]


사용자의 질문과 가장 유사한 문서를 검색하는 함수입니다.

- **query → embedding**
    - 질문을 벡터로 변환
- **index.search**
    - 가장 가까운 문서 top_k개 검색
- **indices**
    - 유사한 문서의 인덱스 반환

즉, 질문을 벡터로 변환 → 문서들과 거리 계산 → 가장 가까운 문서 검색을 수행하는 함수인것이죠.

**⭐️ 개념 정리**

- retrieve 함수의 전체 흐름을 정리해주세요.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


**실행 결과 확인**

아래 셀을 실행해, 겹치는 단어가 거의 없는 질문으로도 관련 문서를 찾는지 확인해주세요. `top_k` 값을 바꿔 다시 실행해봐도 좋습니다.


In [ ]:
# [제공 코드] retrieve 함수로 문서를 검색합니다.
for doc in retrieve("돈은 언제 돌려받을 수 있나요?", top_k=2):
    print(doc)


**생각해보기:**

- 출력된 문서가 질문과 관련이 있나요?
- top_k 값은 어떤 역할을 하나요? 값을 너무 작게, 또는 너무 크게 정하면 어떤 문제가 생길까요?

예상과 다른 결과가 나왔다면, 실제로 관찰한 결과 그대로 정리해주세요.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


### **7. 답변을 생성할 LLM 불러오기**


In [ ]:
# [제공 코드] 답변을 생성할 LLM을 불러옵니다.
from transformers import pipeline
from transformers.utils import logging
logging.set_verbosity_error()  # 불필요한 경고 메시지 숨기기

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

def generate(prompt):
    messages = [
        {"role": "system", "content": "너는 쇼핑몰 고객센터 상담원이야. 고객의 문의에 친절하고 상냥한 말투로 답변해줘."},
        {"role": "user", "content": prompt}
    ]
    output = generator(messages, max_new_tokens=200, do_sample=False)
    return output[0]["generated_text"][-1]["content"]


RAG의 마지막 단계는 **생성(Generation)** 입니다. 검색한 문서를 바탕으로 자연스러운 문장의 답변을 만들어야 하므로, 문장을 생성할 수 있는 LLM이 필요합니다.

Mission1에서 사용한 BERT는 문장을 이해하고 분류는 잘하지만, 답변 문장을 직접 만들어내지는 못합니다. 그래서 이번에는 문장을 생성할 수 있는 오픈소스 LLM `Qwen2.5-3B-Instruct`를 불러올 거예요. 파라미터가 약 30억 개(3B)인 작은 모델이라 API 키 없이 Colab GPU나 로컬 환경에서도 실행할 수 있답니다. 이름의 `Instruct`는 사용자의 지시와 질문에 맞춰 답하도록 추가 학습된 버전이라는 뜻입니다.

- `messages`: llm 모델에게 전달할 대화 형식의 입력 (`system`: 모델의 역할 지정, `user`: 실제 질문)
- `max_new_tokens`: 답변을 생성할 최대 토큰 수
- `do_sample=False`: 매번 가장 확률이 높은 토큰을 선택하여, 같은 질문에는 같은 답변이 나오도록 설정


### **8. RAG 전체 파이프라인 이해하기**

**[⚓️ 미션]** `retrieved`의 빈칸을 채워, 질문과 관련된 문서를 검색해주세요.

**HINT:**  6단계에서 만든 `retrieve()`에 `query`를 전달하고, `top_k`는 2로 지정해보세요.


In [ ]:
def rag_pipeline(query):
    print(f"[질문]: {query}")

    retrieved = None  # TODO: 질문과 관련된 문서 2개를 검색해주세요.
    context = "\n".join(retrieved)
    print(f"[검색된 문서]:\n{context}")

    prompt = f"""아래 [문서]는 우리 쇼핑몰의 배송/환불 정책이야.
[문서]의 내용을 근거로 고객의 [질문]에 1~2문장으로 답변해줘.
[문서]에서 질문과 관련된 내용을 찾을 수 없을 때에는,  "관련된 내용을 [배송/환불 정책]에서 찾을 수 없습니다."라고 답변해줘.

[문서]
{context}

[질문]
{query}"""

    answer = generate(prompt)
    print(f"[답변]: {answer}\n")
    print("-" * 60)


RAG의 핵심 파이프라인을 구성하는 단계입니다.

처음에 설명드렸던 RAG 파이프라인과 함께 설명드리자면, 

```markdown
질문 입력
→ query

↓
관련 문서 검색
→ retrieved = retrieve(query, top_k=2)

↓
검색된 문서를 Context로 생성
→ context = "\n".join(retrieved)

↓
질문 + Context를 함께 모델에 입력
→ prompt = f"...[문서] {context} [질문] {query}"

↓
최종 결과 생성
→ answer = generate(prompt)
```

검색된 문서를 프롬프트에 함께 넣어 LLM의 입력을 보강(Augmented)하기 때문에, 이 방식을 **검색 증강 생성(Retrieval-Augmented Generation)** 이라고 부릅니다.

<img src="imgs/rag_pipeline.png" width="600" />

(출처: https://yozm.wishket.com/magazine/detail/2828)

**⭐️ 개념 정리**

- RAG 전체 구조를 간단히 정리해주세요.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


### **9. RAG 실행하기**


In [ ]:
# [제공 코드] RAG 파이프라인을 실행합니다.
queries = [
    "환불은 며칠 안에 돼요?",
    "제주도도 무료배송인가요?",
    "택을 뗐는데 반품할 수 있나요?"
]

for q in queries:
    rag_pipeline(q)


다양한 질문에 대해 RAG 파이프라인을 실행해봅시다.

질문과 관련된 정책 문서를 먼저 검색한 뒤, 검색된 문서를 바탕으로 LLM이 답변을 생성하게 됩니다!

### **10. RAG 없이 질문해보기**


In [ ]:
# [제공 코드] 같은 질문을 문서 검색 없이 LLM에 입력합니다.
for q in queries:
    print(f"[질문]: {q}")
    print(f"[답변]: {generate(q)}\n")
    print("-" * 60)


이번에는 문서 검색 없이, 같은 질문을 LLM에게 그대로 물어봅시다.

LLM은 저희 쇼핑몰의 정책을 학습한 적이 없기 때문에, 일반적인 쇼핑몰 정책을 추측해서 답하거나 사실과 다른 내용을 그럴듯하게 지어낼 수 있습니다. 이렇게 모델이 사실이 아닌 내용을 그럴듯하게 생성하는 현상을 `환각(Hallucination)`이라고 합니다.

**생각해보기:**

- RAG를 사용했을 때와 사용하지 않았을 때의 답변을 비교해보면, 어떤 차이가 있나요?
- RAG는 어떤 상황에서 사용하면 좋을까요?
- Fine-Tuning과 RAG는 어떤 차이가 있을까요? 


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**


### 마무리

- `TODO`의 핵심 코드를 작성하고 위에서 아래로 실행해주세요.
- 출력을 확인하고, 실행 결과가 의미하는 바를 정리해주세요.
- **⭐️ 개념 정리**와 **생각해보기**는 노트북의 답변 셀에 정리해주세요.
- 알게 된 점, 막힌 지점, 해결 방법을 담아 300자 이상 WIL을 작성해주세요.
- 실습한 코드를 `.ipynb`로 저장하여, github에 올려주세요.

